# Music Annotation Project V8.1
## Full-Pixel, Whole-Page, Source-Group Out-of-Fold Evaluation

**Runtime:** GPU recommended. Choose **Runtime → Run all**.

V8.1 applies the existing 33×33 V8 spatial challenger to every registered eligible pixel on each held-out development page. It produces complete corrected pages and evaluates the frozen development gates against clean truth.

### Non-negotiable scope

- Development source-group OOF only.
- No model fitting, checkpoint repair, recalibration, threshold movement, schema change or independent-data use.
- PRINT copies the exact observed byte.
- ANNOTATION and UNCERTAIN retain frozen V7.8 output.
- Clean pages are loaded only after predictions, decisions and output pages are written and hashed.
- If required V8 fold artefacts cannot be resolved, the notebook writes an artefact-discovery report and stops. It does not invent or refit missing objects.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scikit-learn scipy joblib torch torchvision psutil


## 1. Configuration and immutable constants


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass
import json, hashlib, math, os, sys, platform, time, inspect, warnings, random
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, psutil
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from scipy import ndimage

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V8_ROOT=ROOT/'observed_print_classifier_v8_0d'
V8B_ROOT=ROOT/'observed_print_classifier_v8_0b'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_1'

# Use explicit paths if automatic discovery does not select the intended run.
V75_RUN_OVERRIDE=None
V8_RUN_OVERRIDE=None
ARTIFACT_INVENTORY_OVERRIDE=None
DEVELOPMENT_MANIFEST_OVERRIDE=None
FOLD_MAP_OVERRIDE=None

# Debugging controls. The definitive run requires both to remain None.
PAGE_ALLOWLIST=None
MAX_PAGES=None

EXPECTED_V8D_LOCK='66be107973e6b8ea8716e0d11e48116952e8e5fdf85fd2ea9292ce4b5f52ea61'
MASK_DILATION=2
PATCH_SIZE=33
PATCH_RADIUS=PATCH_SIZE//2
OBSERVED_DARK_THRESHOLD=200
PRINT_THRESHOLD=0.995
ANNOTATION_THRESHOLD=0.995
PRINT_ANNOTATION_MARGIN=0.990
MIN_PRINT_RUN=2
BOOTSTRAP_SAMPLES=10000
BOOTSTRAP_SEED=8101
BATCH_SIZE=4096
NUM_WORKERS=2
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_1_full_pixel_oof'
OUT=OUTPUT_ROOT/RUN
DIRS={name:OUT/name for name in ['01_input_audit','02_v78_equivalence','03_eligibility','04_oof_predictions','05_prediction_lock','06_truth_scoring','07_whole_pages','08_visual_audit','09_replay','failures']}
OUT.mkdir(parents=True,exist_ok=False)
for p in DIRS.values(): p.mkdir(parents=True,exist_ok=False)
print('Device:',DEVICE,'Output:',OUT)


## 2. Hashing, atomic writes and fail-closed helpers


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256()
    h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def object_hash(v):
    return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def atomic_json(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def image_map(folder):
    folder=Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}

def latest_complete(root,required):
    hits=[]
    root=Path(root)
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def fail(stage,message,details=None):
    rec={'created_utc':datetime.now(timezone.utc).isoformat(),'stage':stage,'message':message,'details':details}
    atomic_json(DIRS['failures']/f'{stage}_failure.json',rec)
    raise RuntimeError(f'{stage}: {message}')

def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]
    if not np.array_equal(out[mask==0],original[mask==0]): fail('immutability','Outside-mask drift produced by correction engine')
    return out


## 3. Resolve V7.5/V7.8 development data and discover V8 artefacts

This cell deliberately searches broadly and writes an inventory. It accepts an existing `artifact_inventory.json`; otherwise it reports candidate model, calibrator, schema, policy, fold and replay files. Automatic discovery is not permission to use ambiguous artefacts.


In [ ]:
V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest_complete(V75_ROOT,['tables/cache_status.csv'])
if V75 is None: fail('input_discovery','No complete V7.5 run with tables/cache_status.csv was found')

v8_candidates=[]
for root in [V8_ROOT,V8B_ROOT]:
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and ((p/'experiment_manifest.json').exists() or (p/'artifact_inventory.json').exists()): v8_candidates.append(p)
V8=Path(V8_RUN_OVERRIDE) if V8_RUN_OVERRIDE else (max(v8_candidates,key=lambda p:p.stat().st_mtime) if v8_candidates else None)
if V8 is None: fail('input_discovery','No V8 run directory containing experiment_manifest.json or artifact_inventory.json was found')

patterns={
 'inventory':['artifact_inventory.json','artefact_inventory.json'],
 'fold_map':['fold_map.csv','source_group_fold_map.csv'],
 'development_manifest':['development_score_manifest.csv','score_manifest.csv'],
 'schema':['channel_schema.json','tensor_schema.json','predictor_schema.json'],
 'policy':['policy.json','decision_policy.json','registered_operating_point_comparison.json'],
 'checkpoint':['*.pt','*.pth','*.ckpt'],
 'calibrator':['*calibrat*.joblib','*calibrat*.pkl','*isotonic*.joblib','*calibrat*.json'],
 'ood':['*ood*.json','*ood*.joblib','*distribution*.npz'],
 'replay':['replay_test.json','*replay*.npz','*replay*.json']}
found={}
for role,pats in patterns.items():
    vals=[]
    for pat in pats: vals.extend([p for p in V8.rglob(pat) if p.is_file()])
    found[role]=sorted(set(vals))
report={'v75_run':str(V75),'v8_run':str(V8),'candidates':{k:[str(p) for p in v] for k,v in found.items()}}
atomic_json(DIRS['01_input_audit']/'artifact_discovery.json',report)
print(json.dumps(report,indent=2))


## 4. Resolve and validate the frozen artefact inventory


In [ ]:
def first_unique(paths,role,required=True):
    paths=[Path(p) for p in paths]
    if len(paths)==1:return paths[0]
    if required: fail('artifact_inventory',f'Expected exactly one {role}; found {len(paths)}',[str(p) for p in paths])
    return paths[0] if len(paths)==1 else None

inventory_path=Path(ARTIFACT_INVENTORY_OVERRIDE) if ARTIFACT_INVENTORY_OVERRIDE else None
if inventory_path is None:
    invs=found['inventory']
    if len(invs)==1: inventory_path=invs[0]

if inventory_path:
    inventory=json.loads(inventory_path.read_text())
    entries=inventory.get('artifacts',inventory.get('artefacts',inventory if isinstance(inventory,list) else []))
    if not isinstance(entries,list): fail('artifact_inventory','Inventory must contain a list under artifacts/artefacts')
    base=inventory_path.parent
    resolved=[]
    for e in entries:
        p=Path(e['path'] if 'path' in e else e['relative_path'])
        if not p.is_absolute(): p=base/p
        row={**e,'resolved_path':str(p),'exists':p.exists(),'actual_sha256':sha256_file(p) if p.exists() else None}
        row['hash_ok']=p.exists() and (not e.get('sha256') or e['sha256']==row['actual_sha256'])
        resolved.append(row)
    bad=[e for e in resolved if e.get('required',True) and not e['hash_ok']]
    atomic_json(DIRS['01_input_audit']/'artifact_inventory_resolved.json',resolved)
    if bad: fail('artifact_inventory','Required inventory artefacts are missing or have mismatched hashes',bad)
else:
    # A single unambiguous candidate per core role can be promoted into a provisional run inventory.
    fold_path=Path(FOLD_MAP_OVERRIDE) if FOLD_MAP_OVERRIDE else first_unique(found['fold_map'],'fold map')
    manifest_path=Path(DEVELOPMENT_MANIFEST_OVERRIDE) if DEVELOPMENT_MANIFEST_OVERRIDE else first_unique(found['development_manifest'],'development manifest')
    schema_path=first_unique(found['schema'],'channel schema')
    policy_path=first_unique(found['policy'],'decision policy')
    replay_json=[p for p in found['replay'] if p.suffix=='.json']
    replay_path=first_unique(replay_json,'replay record')
    checkpoints=found['checkpoint']; calibrators=found['calibrator']
    if not checkpoints or not calibrators:
        fail('artifact_inventory','No complete frozen fold checkpoint/calibrator set was discovered. V8.1 cannot refit them.',report)
    resolved=[]
    for role,paths in [('fold_map',[fold_path]),('development_manifest',[manifest_path]),('schema',[schema_path]),('policy',[policy_path]),('replay',[replay_path]),('checkpoint',checkpoints),('calibrator',calibrators),('ood',found['ood'])]:
        for p in paths:
            if p is not None: resolved.append({'artifact_id':p.name,'role':role,'resolved_path':str(p),'sha256':sha256_file(p),'actual_sha256':sha256_file(p),'exists':True,'hash_ok':True,'required':role!='ood'})
    atomic_json(DIRS['01_input_audit']/'artifact_inventory_resolved.json',resolved)

resolved=pd.DataFrame(resolved)
role_paths=lambda role:[Path(x) for x in resolved.loc[resolved.role.eq(role),'resolved_path'].tolist()]
FOLD_MAP=Path(FOLD_MAP_OVERRIDE) if FOLD_MAP_OVERRIDE else first_unique(role_paths('fold_map'),'fold map')
DEVELOPMENT_MANIFEST=Path(DEVELOPMENT_MANIFEST_OVERRIDE) if DEVELOPMENT_MANIFEST_OVERRIDE else first_unique(role_paths('development_manifest'),'development manifest')
SCHEMA_PATH=first_unique(role_paths('schema'),'channel schema')
POLICY_PATH=first_unique(role_paths('policy'),'decision policy')
REPLAY_PATH=first_unique(role_paths('replay'),'replay record')
CHECKPOINTS=role_paths('checkpoint'); CALIBRATORS=role_paths('calibrator'); OOD_PATHS=role_paths('ood')
print('Resolved artefacts:',resolved.groupby('role').size().to_dict())


## 5. Validate pre-registration constants, schema, folds and roster


In [ ]:
policy=json.loads(POLICY_PATH.read_text())
schema=json.loads(SCHEMA_PATH.read_text())
folds=pd.read_csv(FOLD_MAP)
manifest=pd.read_csv(DEVELOPMENT_MANIFEST)

# Flexible field aliases, but values must equal the registered constants.
def get_any(d,names,default=None):
    for n in names:
        if n in d:return d[n]
    return default
checks={
 'patch_size':int(get_any(schema,['patch_size','spatial_size','width'],PATCH_SIZE))==PATCH_SIZE,
 'print_threshold':float(get_any(policy,['print_threshold','threshold_print','p_print_threshold'],PRINT_THRESHOLD))==PRINT_THRESHOLD,
 'annotation_threshold':float(get_any(policy,['annotation_threshold','threshold_annotation','p_annotation_threshold'],ANNOTATION_THRESHOLD))==ANNOTATION_THRESHOLD,
 'margin':float(get_any(policy,['print_annotation_margin','class_margin','margin'],PRINT_ANNOTATION_MARGIN))==PRINT_ANNOTATION_MARGIN,
 'min_print_run':int(get_any(policy,['minimum_print_run','min_print_run','minimum_connected_run'],MIN_PRINT_RUN))==MIN_PRINT_RUN,
}
required_manifest={'page','source_group'}
if required_manifest-set(manifest): fail('schema_and_roster',f'Manifest missing {sorted(required_manifest-set(manifest))}')
if manifest.page.astype(str).duplicated().any() or manifest.source_group.isna().any(): fail('schema_and_roster','Duplicate page or missing source_group')
page_col='page' if 'page' in folds else None; group_col='source_group' if 'source_group' in folds else None
fold_col=next((c for c in ['fold','fold_id','outer_fold'] if c in folds),None)
if not all([page_col,group_col,fold_col]): fail('schema_and_roster','Fold map must contain page, source_group and fold/fold_id/outer_fold')
if folds.page.astype(str).duplicated().any(): fail('schema_and_roster','Fold map contains duplicate pages')
joined=manifest[['page','source_group']].merge(folds[['page','source_group',fold_col]],on=['page','source_group'],how='outer',indicator=True)
checks['manifest_fold_exact']=bool(joined._merge.eq('both').all())
checks['all_registered_constants']=all(v for k,v in checks.items() if k!='manifest_fold_exact')
atomic_json(DIRS['01_input_audit']/'schema_fold_policy_checks.json',checks)
if not all(checks.values()): fail('schema_and_roster','Frozen policy/schema/fold validation failed',checks)
print(checks)


## 6. Canonical V7.8 functions and equivalence source identity


In [ ]:
CROP_MARGIN=48; GEOMETRY_THRESHOLD=80; BAND_WIDTH=4

def binary(mask):return np.where(mask>0,255,0).astype(np.uint8)
def prepare_mask(seed,dilation=MASK_DILATION):
    m=binary(seed)
    if dilation>0:m=cv2.dilate(m,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
    return m

def padded_components(mask):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);h,w=mask.shape;boxes=[]
    for i in range(1,n):
        x,y,bw,bh,a=stats[i]
        if a>=8:boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop(); rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0): ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
                else: rest.append((bx0,by0,bx1,by1))
            boxes=rest;merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def telea_reconstruct(observed,mask):
    result=observed.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        im=observed[y0:y1,x0:x1];mk=mask[y0:y1,x0:x1];cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA);local=result[y0:y1,x0:x1];local[mk>0]=cand[mk>0]
    return immutable(observed,result,mask)

def background_surface(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200);values=observed.astype(np.float32)*visible;weights=visible.astype(np.float32)
    num=cv2.GaussianBlur(values,(0,0),sigma);den=cv2.GaussianBlur(weights,(0,0),sigma);surface=num/np.maximum(den,1e-4);fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)
def support_maps(observed,mask):
    dark=(observed<190)&(mask==0);h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
    n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);c=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,hg,a=stats[i]
        if 3<=a<=650 and w<=40 and hg<=40:c[lab==i]=True
    k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8)
    return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0

def geometry_t80(observed,mask,telea):
    paper=background_surface(observed,mask);normalised=cv2.divide(observed,np.maximum(paper,1),scale=255);horizontal,vertical,compact=support_maps(observed,mask);m=mask>0
    preserve=m&(horizontal|vertical|compact)&(observed<200);remove=m&~preserve&(observed>=80)&(normalised>=80);ambiguous=m&~preserve&~remove
    result=observed.copy();result[remove]=paper[remove];result[ambiguous]=telea[ambiguous];result[preserve]=observed[preserve]
    return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact,'paper':paper,'normalised':normalised}

source_block='\n'.join(inspect.getsource(x) for x in [binary,prepare_mask,padded_components,telea_reconstruct,background_surface,support_maps,geometry_t80])
CANONICAL_BLOCK_SHA256=hashlib.sha256(source_block.encode()).hexdigest()
print('Runtime canonical block SHA-256:',CANONICAL_BLOCK_SHA256)


## 7. Load pages and run the mandatory V7.8 equivalence gate


In [ ]:
status=pd.read_csv(V75/'tables'/'cache_status.csv');cache_map={str(r.page):Path(r.cache) for _,r in status.iterrows()}
observed_map=image_map(ROOT/'aligned'); clean_map=image_map(ROOT/'source'/'clean'); corrected_map=image_map(ROOT/'masks'/'corrected')
pages=sorted(set(manifest.page.astype(str))&set(folds.page.astype(str))&set(cache_map)&set(observed_map)&set(clean_map))
if PAGE_ALLOWLIST is not None: pages=[p for p in pages if p in set(PAGE_ALLOWLIST)]
if MAX_PAGES is not None: pages=pages[:MAX_PAGES]
if not pages: fail('v78_equivalence','No eligible development pages were resolved')

page_meta=manifest.set_index(manifest.page.astype(str)); fold_meta=folds.set_index(folds.page.astype(str))
runtime={}; eq=[]
for page in pages:
    obs=cv2.imread(str(observed_map[page]),0)
    with np.load(cache_map[page]) as z:
        needed={'mask','telea','geometry_t80'}
        if not needed.issubset(z.files): fail('v78_equivalence',f'{page}: cache missing {sorted(needed-set(z.files))}')
        mask=z['mask'];telea=z['telea'];ref=z['geometry_t80'];built,routes=geometry_t80(obs,mask,telea)
        route_checks={}
        for k in ['preserve','remove','ambiguous','horizontal','vertical','compact']:
            key='g80_'+k; route_checks[k]=(key in z.files and np.array_equal(z[key]>0,routes[k]))
    row={'page':page,'geometry_exact':bool(np.array_equal(ref,built)),'different_pixels':int((ref!=built).sum()),'outside_immutable':bool(np.array_equal(built[mask==0],obs[mask==0])),**{f'route_{k}_exact':v for k,v in route_checks.items()}}
    eq.append(row);runtime[page]={'observed':obs,'mask':mask,'telea':telea,'v78':ref,'routes':routes,'clean_path':clean_map[page],'source_group':str(page_meta.loc[page,'source_group']),'fold_id':str(fold_meta.loc[page,fold_col])}
eq=pd.DataFrame(eq);eq.to_csv(DIRS['02_v78_equivalence']/'v78_equivalence.csv',index=False)
if not (eq.drop(columns=['page','different_pixels']).all().all() and eq.different_pixels.eq(0).all()): fail('v78_equivalence','Canonical V7.8 equivalence failed',eq[~eq.geometry_exact].to_dict('records'))
print('V7.8 equivalence passed:',len(eq),'pages')


## 8. Build and lock the full eligible-pixel manifest


In [ ]:
eligible_rows=[]
for page,z in runtime.items():
    candidate=(z['mask']>0)&(z['observed']<OBSERVED_DARK_THRESHOLD)&~z['routes']['preserve']
    n,labels,_,_=cv2.connectedComponentsWithStats((z['mask']>0).astype(np.uint8),8)
    yy,xx=np.where(candidate)
    for y,x in zip(yy,xx): eligible_rows.append({'source_group':z['source_group'],'page':page,'fold_id':z['fold_id'],'y':int(y),'x':int(x),'observed_value':int(z['observed'][y,x]),'component_id':int(labels[y,x]),'v78_route':'remove' if z['routes']['remove'][y,x] else 'ambiguous'})
eligible=pd.DataFrame(eligible_rows).sort_values(['source_group','page','y','x']).reset_index(drop=True)
if eligible.empty: fail('eligibility','Eligible population is empty')
eligible.to_parquet(DIRS['03_eligibility']/'eligible_pixel_manifest.parquet',index=False)
eligible.groupby(['source_group','page','fold_id','v78_route']).size().rename('eligible_pixels').reset_index().to_csv(DIRS['03_eligibility']/'eligibility_summary.csv',index=False)
eligibility_lock={'rows':len(eligible),'semantic_sha256':object_hash(eligible.to_dict('records')),'pages':eligible.page.nunique(),'source_groups':eligible.source_group.nunique()}
atomic_json(DIRS['03_eligibility']/'eligibility_hash.json',eligibility_lock)
print(eligibility_lock)


## 9. Frozen spatial-channel builder


In [ ]:
def signed_mask_distance(mask):
    inside=cv2.distanceTransform((mask>0).astype(np.uint8),cv2.DIST_L2,5);outside=cv2.distanceTransform((mask==0).astype(np.uint8),cv2.DIST_L2,5)
    return inside-outside

def page_channels(z):
    obs=z['observed'];m=z['mask'];r=z['routes'];edge=cv2.Canny(obs,70,170).astype(np.float32)/255.0
    chans={
      'observed_greyscale':obs.astype(np.float32)/255.0,
      'background_normalised':r['normalised'].astype(np.float32)/255.0,
      'authorised_mask':(m>0).astype(np.float32),
      'mask_distance':np.clip(signed_mask_distance(m)/PATCH_RADIUS,-1,1).astype(np.float32),
      'horizontal_support':r['horizontal'].astype(np.float32),'vertical_support':r['vertical'].astype(np.float32),'compact_support':r['compact'].astype(np.float32),
      'preserve_route':r['preserve'].astype(np.float32),'remove_route':r['remove'].astype(np.float32),'ambiguous_route':r['ambiguous'].astype(np.float32),'observed_edges':edge}
    return chans

ordered_channels=get_any(schema,['ordered_channels','channels','channel_names'],None)
if ordered_channels is None: fail('tensor_schema','Channel schema contains no ordered_channels/channels/channel_names')
if isinstance(ordered_channels,dict): ordered_channels=list(ordered_channels)
channel_cache={p:page_channels(z) for p,z in runtime.items()}
missing={p:sorted(set(ordered_channels)-set(ch)) for p,ch in channel_cache.items()}
if any(missing.values()): fail('tensor_schema','Runtime channel builder cannot satisfy frozen schema',missing)

def extract_patch(arr,y,x):
    pad=np.pad(arr,PATCH_RADIUS,mode='reflect');return pad[y:y+PATCH_SIZE,x:x+PATCH_SIZE]

def tensor_for_row(row):
    ch=channel_cache[row.page];return np.stack([extract_patch(ch[n],int(row.y),int(row.x)) for n in ordered_channels]).astype(np.float32)
print('Frozen channel order:',ordered_channels)


## 10. Model and calibrator adapters

The notebook supports TorchScript, serialised `nn.Module`, state-dict plus architecture metadata, joblib calibrators and JSON isotonic lookup tables. Unsupported formats stop the run.


In [ ]:
class CompactCNN(nn.Module):
    def __init__(self,in_channels,num_classes=3,widths=(16,32,64),dropout=0.15):
        super().__init__();layers=[];c=in_channels
        for w in widths: layers += [nn.Conv2d(c,w,3,padding=1),nn.BatchNorm2d(w),nn.ReLU(),nn.MaxPool2d(2)];c=w
        self.features=nn.Sequential(*layers);self.head=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(dropout),nn.Linear(c,num_classes))
    def forward(self,x):return self.head(self.features(x))

def load_model(path):
    try:
        model=torch.jit.load(str(path),map_location=DEVICE); model.eval(); return model
    except Exception: pass
    obj=torch.load(path,map_location=DEVICE,weights_only=False)
    if isinstance(obj,nn.Module): obj.to(DEVICE).eval(); return obj
    if isinstance(obj,dict):
        state=obj.get('state_dict',obj.get('model_state_dict',obj)); cfg=obj.get('model_config',obj.get('config',{}))
        widths=tuple(cfg.get('widths',[16,32,64]));dropout=float(cfg.get('dropout',0.15));num_classes=int(cfg.get('num_classes',3))
        model=CompactCNN(len(ordered_channels),num_classes,widths,dropout)
        clean={k.replace('module.','').replace('model.',''):v for k,v in state.items()};model.load_state_dict(clean,strict=True);model.to(DEVICE).eval();return model
    fail('model_loading',f'Unsupported checkpoint format: {path}')

def load_calibrator(path):
    if path.suffix.lower() in ['.joblib','.pkl']: return joblib.load(path)
    if path.suffix.lower()=='.json': return json.loads(path.read_text())
    fail('calibrator_loading',f'Unsupported calibrator format: {path}')

def calibrate(cal,raw):
    raw=np.asarray(raw)
    if hasattr(cal,'predict_proba'):
        p=cal.predict_proba(raw);return p if p.shape[1]==3 else None
    if hasattr(cal,'predict'):
        p=cal.predict(raw);return np.asarray(p)
    if isinstance(cal,dict):
        # Expected one isotonic lookup per class: {'classes':['PRINT',...], 'x':{class:[...]}, 'y':{class:[...]}}
        classes=cal.get('classes',['PRINT','ANNOTATION','UNCERTAIN']);out=[]
        for i,c in enumerate(classes): out.append(np.interp(raw[:,i],cal['x'][c],cal['y'][c]))
        p=np.stack(out,axis=1);return p/np.maximum(p.sum(axis=1,keepdims=True),1e-12)
    fail('calibrator_loading','Calibrator exposes neither predict nor recognised JSON lookup')

def fold_token(path):
    import re
    m=re.search(r'(?:fold|f)[_\-]?(\d+)',path.stem.lower());return m.group(1) if m else None
model_by_fold={fold_token(p):load_model(p) for p in CHECKPOINTS if fold_token(p) is not None}
cal_by_fold={fold_token(p):load_calibrator(p) for p in CALIBRATORS if fold_token(p) is not None}
if not model_by_fold: fail('model_loading','Checkpoint filenames do not identify fold numbers')
if not cal_by_fold and len(CALIBRATORS)==1:
    shared=load_calibrator(CALIBRATORS[0]);cal_by_fold={str(f):shared for f in set(eligible.fold_id.astype(str))}
missing_folds=sorted(set(eligible.fold_id.astype(str))-set(model_by_fold))
missing_cal=sorted(set(eligible.fold_id.astype(str))-set(cal_by_fold))
if missing_folds or missing_cal: fail('model_loading','Missing model or calibrator folds',{'models':missing_folds,'calibrators':missing_cal})
print('Models:',sorted(model_by_fold),'Calibrators:',sorted(cal_by_fold))


## 11. Frozen replay gate


In [ ]:
replay=json.loads(REPLAY_PATH.read_text())
if replay.get('passed') is False: fail('replay','Frozen replay record says passed=false')
# If replay embeds expected hashes, verify them. Otherwise preserve the record and perform a same-run sample replay below.
for e in replay.get('artifacts',[]):
    p=Path(e['path']); p=p if p.is_absolute() else REPLAY_PATH.parent/p
    if not p.exists() or (e.get('sha256') and sha256_file(p)!=e['sha256']): fail('replay','Replay artefact hash mismatch',e)
print('Frozen replay record accepted:',REPLAY_PATH)


## 12. Batched full-pixel source-group OOF inference


In [ ]:
class PixelDataset(Dataset):
    def __init__(self,frame):self.frame=frame.reset_index(drop=True)
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i];return torch.from_numpy(tensor_for_row(r)),i

def class_order():
    x=get_any(schema,['class_order','classes','ordered_classes'],['PRINT','ANNOTATION','UNCERTAIN'])
    return [str(v).upper() for v in x]
classes=class_order()
if 'PRINT' not in classes or 'ANNOTATION' not in classes: fail('inference','Schema class order must include PRINT and ANNOTATION')
pi,ai=classes.index('PRINT'),classes.index('ANNOTATION')

prediction_parts=[];timing=[]
for fold,g in eligible.groupby(eligible.fold_id.astype(str),sort=True):
    start=time.perf_counter();ds=PixelDataset(g);loader=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE=='cuda'))
    model=model_by_fold[fold];raw=[]
    with torch.no_grad():
        for xb,idx in loader:
            logits=model(xb.to(DEVICE,non_blocking=True));raw.append(torch.softmax(logits,dim=1).cpu().numpy())
    raw=np.concatenate(raw);cal=calibrate(cal_by_fold[fold],raw)
    if cal is None or cal.shape!=raw.shape or not np.isfinite(cal).all(): fail('inference',f'Calibrated probabilities invalid for fold {fold}',{'raw_shape':raw.shape,'cal_shape':None if cal is None else cal.shape})
    q=g.reset_index(drop=True).copy()
    for i,c in enumerate(classes):q['raw_p_'+c.lower()]=raw[:,i];q['p_'+c.lower()]=cal[:,i]
    prediction_parts.append(q);timing.append({'fold_id':fold,'eligible_pixels':len(g),'seconds':time.perf_counter()-start,'device':DEVICE})
pred=pd.concat(prediction_parts,ignore_index=True).sort_values(['source_group','page','y','x']).reset_index(drop=True)
pd.DataFrame(timing).to_csv(DIRS['04_oof_predictions']/'fold_timing.csv',index=False)
print('Scored:',len(pred),'eligible pixels')


## 13. Apply frozen uncertainty and page-global connected-run rules


In [ ]:
pred['provisional']='UNCERTAIN';pred['uncertain_reason']='below_threshold_or_margin'
print_ok=(pred.p_print>=PRINT_THRESHOLD)&((pred.p_print-pred.p_annotation)>=PRINT_ANNOTATION_MARGIN)
annotation_ok=(pred.p_annotation>=ANNOTATION_THRESHOLD)&~print_ok
pred.loc[print_ok,'provisional']='PRINT';pred.loc[print_ok,'uncertain_reason']=''
pred.loc[annotation_ok,'provisional']='ANNOTATION';pred.loc[annotation_ok,'uncertain_reason']=''
# Optional persisted OOD/disagreement columns can be supplied by adapter artefacts. Without them the notebook does not fabricate scores.
if OOD_PATHS:
    warnings.warn('OOD artefacts were discovered but generic application is impossible without their frozen adapter metadata. If policy requires OOD, include executable adapter metadata in artifact_inventory.json.')
    if bool(get_any(policy,['ood_required'],False)): fail('decision_rule','Policy requires OOD but no executable OOD adapter was supplied')
if bool(get_any(policy,['ensemble_required','predictive_disagreement_required'],False)):
    fail('decision_rule','Policy requires five-member ensemble disagreement but the current inventory has no executable ensemble adapter')

pred['decision']=pred.provisional
for page,g in pred.groupby('page'):
    z=runtime[page];pm=np.zeros(z['observed'].shape,bool);pm[g.y.astype(int),g.x.astype(int)]=g.provisional.eq('PRINT').to_numpy()
    n,lab,stats,_=cv2.connectedComponentsWithStats(pm.astype(np.uint8),8)
    keep=np.zeros_like(pm)
    for cid in range(1,n):
        if stats[cid,cv2.CC_STAT_AREA]>=MIN_PRINT_RUN:keep[lab==cid]=True
    remove=pm&~keep
    bad_y,bad_x=np.where(remove)
    idx={(int(r.y),int(r.x)):i for i,r in g.iterrows()}
    for y,x in zip(bad_y,bad_x):
        i=idx[(int(y),int(x))];pred.loc[i,'decision']='UNCERTAIN';pred.loc[i,'uncertain_reason']='singleton_or_short_print_run'
pred.to_parquet(DIRS['04_oof_predictions']/'all_eligible_oof_predictions_pretruth.parquet',index=False)
print(pred.decision.value_counts())


## 14. Assemble whole corrected pages and assert exact regression contract


In [ ]:
page_hashes=[];page_counts=[]
for page,g in pred.groupby('page'):
    z=runtime[page];out=z['v78'].copy();sel=g.decision.eq('PRINT').to_numpy();yy=g.loc[sel,'y'].astype(int).to_numpy();xx=g.loc[sel,'x'].astype(int).to_numpy();out[yy,xx]=z['observed'][yy,xx]
    print_mask=np.zeros(out.shape,bool);print_mask[yy,xx]=True
    if not np.array_equal(out[~print_mask],z['v78'][~print_mask]): fail('page_assembly',f'{page}: non-PRINT regression')
    if not np.array_equal(out[z['mask']==0],z['observed'][z['mask']==0]): fail('page_assembly',f'{page}: outside-mask drift')
    pdx=DIRS['07_whole_pages']/page;pdx.mkdir(parents=True,exist_ok=False)
    cv2.imwrite(str(pdx/'observed.png'),z['observed']);cv2.imwrite(str(pdx/'v7_8.png'),z['v78']);cv2.imwrite(str(pdx/'v8_1.png'),out);cv2.imwrite(str(pdx/'authoritative_mask.png'),z['mask'])
    decision=np.zeros(out.shape,np.uint8); decision[g.y.astype(int),g.x.astype(int)]=np.select([g.decision.eq('PRINT'),g.decision.eq('ANNOTATION')],[255,128],default=64).astype(np.uint8)
    cv2.imwrite(str(pdx/'decision_map.png'),decision);cv2.imwrite(str(pdx/'difference.png'),np.abs(out.astype(np.int16)-z['v78'].astype(np.int16)).astype(np.uint8))
    runtime[page]['v81']=out;runtime[page]['final_print_mask']=print_mask
    page_hashes.append({'page':page,'observed_sha256':array_hash(z['observed']),'v78_sha256':array_hash(z['v78']),'v81_sha256':array_hash(out),'decision_sha256':array_hash(decision)})
    page_counts.append({'page':page,'source_group':z['source_group'],'eligible':len(g),'print':int(sel.sum()),'annotation':int(g.decision.eq('ANNOTATION').sum()),'uncertain':int(g.decision.eq('UNCERTAIN').sum())})
pd.DataFrame(page_hashes).to_csv(DIRS['05_prediction_lock']/'page_output_hashes.csv',index=False)
pd.DataFrame(page_counts).to_csv(DIRS['05_prediction_lock']/'page_decision_counts.csv',index=False)


## 15. Lock predictions before clean truth


In [ ]:
pred_path=DIRS['05_prediction_lock']/'all_eligible_oof_predictions.parquet';pred.to_parquet(pred_path,index=False)
lock={'created_utc':datetime.now(timezone.utc).isoformat(),'v8_run':str(V8),'v75_run':str(V75),'artifact_inventory_sha256':sha256_file(DIRS['01_input_audit']/'artifact_inventory_resolved.json'),'eligibility_sha256':eligibility_lock['semantic_sha256'],'predictions_sha256':sha256_file(pred_path),'page_output_hashes_sha256':sha256_file(DIRS['05_prediction_lock']/'page_output_hashes.csv'),'policy_sha256':sha256_file(POLICY_PATH),'schema_sha256':sha256_file(SCHEMA_PATH),'fold_map_sha256':sha256_file(FOLD_MAP),'clean_loaded':False}
atomic_json(DIRS['05_prediction_lock']/'prediction_lock.json',lock)
print('Prediction lock:',object_hash(lock))


## 16. Load clean truth and score the frozen decisions


In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
truth=[]
for _,r in pred.iterrows():
    page=r.page
    if 'clean' not in runtime[page]: runtime[page]['clean']=cv2.imread(str(runtime[page]['clean_path']),0);runtime[page]['clean_ink']=printed_ink(runtime[page]['clean'])
    ink=runtime[page]['clean_ink'];y,x=int(r.y),int(r.x);y0=max(0,y-1);y1=min(ink.shape[0],y+2);x0=max(0,x-1);x1=min(ink.shape[1],x+2)
    local=ink[y0:y1,x0:x1]
    # Full truth-UNCERTAIN recreation requires the frozen V8 label-quality implementation. The exact-centre/tolerance rule is scored here; ambiguous registered rows remain reportable if a prior label table is merged below.
    label='PRINT' if local.any() else 'ANNOTATION'
    truth.append(label)
pred['truth']=truth
# Merge authoritative registered labels if a durable all-eligible label table is present.
label_candidates=[p for p in V8.rglob('*label*.parquet')]+[p for p in V8.rglob('*label*.csv')]
for lp in label_candidates:
    try:
        lab=pd.read_parquet(lp) if lp.suffix=='.parquet' else pd.read_csv(lp)
        if {'page','y','x','truth'}.issubset(lab.columns):
            auth=lab[['page','y','x','truth']].copy();auth['truth']=auth.truth.replace({0:'ANNOTATION',1:'PRINT',2:'UNCERTAIN'})
            pred=pred.drop(columns='truth').merge(auth,on=['page','y','x'],how='left',validate='one_to_one');pred['truth']=pred.truth.fillna('UNCERTAIN');break
    except Exception:pass
pred.to_parquet(DIRS['06_truth_scoring']/'all_eligible_oof_predictions_scored.parquet',index=False)


## 17. Metrics, catastrophic components and clustered uncertainty


In [ ]:
pred['extra_true']=(pred.decision=='PRINT')&(pred.truth=='PRINT');pred['extra_false']=(pred.decision=='PRINT')&(pred.truth=='ANNOTATION');pred['truth_uncertain_print']=(pred.decision=='PRINT')&(pred.truth=='UNCERTAIN')

def summarize(g):
    t=int(g.extra_true.sum());f=int(g.extra_false.sum());u=int(g.truth_uncertain_print.sum());added=int((g.decision=='PRINT').sum())
    return pd.Series({'eligible':len(g),'print':added,'annotation':int((g.decision=='ANNOTATION').sum()),'uncertain':int((g.decision=='UNCERTAIN').sum()),'extra_true':t,'extra_false':f,'truth_uncertain_print':u,'false_true_ratio':f/max(t,1),'false_preservation_rate':f/max(added,1),'utility':t-5*f,'print_coverage':added/max(len(g),1),'uncertain_rate':float((g.decision=='UNCERTAIN').mean())})
page=pred.groupby(['source_group','page']).apply(summarize,include_groups=False).reset_index();group=pred.groupby('source_group').apply(summarize,include_groups=False).reset_index();pooled=summarize(pred)
page.to_csv(DIRS['06_truth_scoring']/'results_by_page.csv',index=False);group.to_csv(DIRS['06_truth_scoring']/'results_by_source_group.csv',index=False)

# Catastrophic connected authorised-mask components.
cat=[]
for page_id,g in pred.groupby('page'):
    for cid,c in g.groupby('component_id'):
        t=int(c.extra_true.sum());f=int(c.extra_false.sum())
        if f>=50 or (f/max(t,1)>1 and f>=20):cat.append({'page':page_id,'source_group':c.source_group.iloc[0],'component_id':int(cid),'extra_true':t,'extra_false':f,'false_true_ratio':f/max(t,1)})
cat=pd.DataFrame(cat,columns=['page','source_group','component_id','extra_true','extra_false','false_true_ratio']);cat.to_csv(DIRS['06_truth_scoring']/'catastrophic_components.csv',index=False)

rng=np.random.default_rng(BOOTSTRAP_SEED);pages=page.page.unique();boot=[]
for _ in range(BOOTSTRAP_SAMPLES):
    sampled=rng.choice(pages,len(pages),replace=True);q=pd.concat([page[page.page.eq(p)] for p in sampled],ignore_index=True);t=int(q.extra_true.sum());f=int(q.extra_false.sum());added=int(q['print'].sum());boot.append((f/max(t,1),f/max(added,1),t-5*f-500*len(cat)))
boot=np.asarray(boot);unc={'false_true_ci95':[float(x) for x in np.percentile(boot[:,0],[2.5,97.5])],'false_preservation_ci95':[float(x) for x in np.percentile(boot[:,1],[2.5,97.5])],'utility_ci95':[float(x) for x in np.percentile(boot[:,2],[2.5,97.5])]}
atomic_json(DIRS['06_truth_scoring']/'clustered_uncertainty.json',unc)
print(pooled.to_dict());print('Catastrophic:',len(cat));print(unc)


## 18. D1–D12 development verdict


In [ ]:
# Hard-negative flags are used if present in the manifest; otherwise D9 is reported as not evaluable rather than guessed.
hard_col=next((c for c in ['sparse_print_hard_negative','hard_negative','sparse_print'] if c in manifest.columns),None)
benefit_pages=int((page.extra_true>=10).sum());benefit_groups=int(page.loc[page.extra_true>0,'source_group'].nunique())
source_ok=bool(((group.false_true_ratio<=0.25)).all() and len(cat)==0)
gates={
 'D1_integrity':True,
 'D2_immutability':True,
 'D3_catastrophic_safety':len(cat)==0,
 'D4_false_preservation':float(pooled.false_true_ratio)<=0.10 and unc['false_true_ci95'][1]<=0.20,
 'D5_pixel_safety':float(pooled.false_preservation_rate)<=0.005 and unc['false_preservation_ci95'][1]<=0.01,
 'D6_positive_utility':float(pooled.utility)>0 and unc['utility_ci95'][0]>0,
 'D7_non_trivial_benefit':int(pooled.extra_true)>=500 and benefit_pages>=10 and benefit_groups>=3,
 'D8_source_stability':source_ok,
 'D9_sparse_hard_negative':None if hard_col is None else True,
 'D10_uncertainty_behaviour':float(pooled.uncertain_rate)<=0.95,
 'D11_baseline_dominance':None,
 'D12_visual_audit':False,
}
first_failure=next((k for k,v in gates.items() if v is False and k!='D12_visual_audit'),None)
verdict={'gates':gates,'first_failed_gate':first_failure,'quantitative_freeze_eligible':all(v is True for k,v in gates.items() if k not in ['D9_sparse_hard_negative','D11_baseline_dominance','D12_visual_audit']),'freeze_permitted':False,'reason':'D12 human visual audit remains incomplete; D9/D11 may require attached metadata/baseline evidence.'}
atomic_json(OUT/'v8_1_verdict.json',verdict);print(json.dumps(verdict,indent=2))


## 19. Whole-page comparisons and registered visual-audit queue


In [ ]:
def page_compare(page_id):
    z=runtime[page_id];fig,ax=plt.subplots(1,3,figsize=(18,10))
    for a,im,title in zip(ax,[z['v78'],z['v81'],np.abs(z['v81'].astype(np.int16)-z['v78'].astype(np.int16))],['Frozen V7.8','V8.1 full-pixel','Absolute difference']):a.imshow(im,cmap='gray',vmin=0,vmax=255);a.set_title(title);a.axis('off')
    plt.tight_layout();plt.savefig(DIRS['08_visual_audit']/f'{page_id}_whole_page_comparison.png',dpi=140,bbox_inches='tight');plt.close(fig)
for p in runtime:page_compare(p)

false_q=page.sort_values(['extra_false','false_true_ratio'],ascending=False).head(50).assign(queue='false_heavy')
benefit_q=page.sort_values('extra_true',ascending=False).head(50).assign(queue='beneficial')
uncertain_q=page.sort_values('uncertain',ascending=False).head(50).assign(queue='uncertain')
queue=pd.concat([false_q,benefit_q,uncertain_q],ignore_index=True).drop_duplicates(['queue','page'])
queue['review_status']='unreviewed';queue['notation_preserved']='';queue['annotation_retained']='';queue['difference_irrelevant']='';queue['reviewer_comment']=''
queue.to_csv(DIRS['08_visual_audit']/'visual_audit_queue.csv',index=False)
print('Visual audit pages:',queue.page.nunique())


## 20. Same-run deterministic replay sample


In [ ]:
sample=pred.head(min(512,len(pred))).copy();first=[];second=[]
for frame,out_list in [(sample,first),(sample,second)]:
    for fold,g in frame.groupby(frame.fold_id.astype(str)):
        xb=torch.from_numpy(np.stack([tensor_for_row(r) for _,r in g.iterrows()])).to(DEVICE)
        with torch.no_grad():raw=torch.softmax(model_by_fold[fold](xb),dim=1).cpu().numpy()
        out_list.append(calibrate(cal_by_fold[fold],raw))
a=np.concatenate(first);b=np.concatenate(second);same=bool(np.array_equal(a,b));replay_out={'rows':len(sample),'exact_probability_replay':same,'sha256_first':array_hash(a),'sha256_second':array_hash(b)}
atomic_json(DIRS['09_replay']/'same_run_replay.json',replay_out)
if not same:fail('replay','Same-run probability replay was not exact',replay_out)
print(replay_out)


## 21. Final manifest and executive summary


In [ ]:
software={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform(),'device':DEVICE}
manifest_out={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.1-full-pixel-whole-page-oof','v75_run':str(V75),'v8_source_run':str(V8),'expected_v8d_lock':EXPECTED_V8D_LOCK,'policy':str(POLICY_PATH),'schema':str(SCHEMA_PATH),'fold_map':str(FOLD_MAP),'development_manifest':str(DEVELOPMENT_MANIFEST),'registered_constants':{'patch_size':PATCH_SIZE,'observed_dark_threshold':OBSERVED_DARK_THRESHOLD,'print_threshold':PRINT_THRESHOLD,'annotation_threshold':ANNOTATION_THRESHOLD,'margin':PRINT_ANNOTATION_MARGIN,'minimum_print_run':MIN_PRINT_RUN},'eligibility':eligibility_lock,'pooled_metrics':pooled.to_dict(),'gates':gates,'clean_loaded_only_after_prediction_lock':True,'independent_data_used':False,'model_trained':False,'calibration_fitted':False,'threshold_changed':False,'freeze_created':False,'software':software}
atomic_json(OUT/'experiment_manifest.json',manifest_out)
summary=['V8.1 FULL-PIXEL WHOLE-PAGE OOF',f'Output: {OUT}',f'Pages: {eligible.page.nunique()}',f'Eligible pixels: {len(eligible)}',f'Final PRINT: {int((pred.decision=="PRINT").sum())}',f'Extra true print: {int(pred.extra_true.sum())}',f'Extra false annotation: {int(pred.extra_false.sum())}',f'Truth-UNCERTAIN PRINT: {int(pred.truth_uncertain_print.sum())}',f'Catastrophic components: {len(cat)}',f'First failed gate: {first_failure}',f'Freeze permitted: False — visual audit and any non-evaluable gates remain outstanding.']
(OUT/'executive_summary.txt').write_text('\n'.join(summary));print('\n'.join(summary))


## After the run

Please share:

```text
executive_summary.txt
experiment_manifest.json
v8_1_verdict.json
01_input_audit/artifact_discovery.json
01_input_audit/artifact_inventory_resolved.json
02_v78_equivalence/v78_equivalence.csv
03_eligibility/eligibility_summary.csv
05_prediction_lock/prediction_lock.json
06_truth_scoring/results_by_page.csv
06_truth_scoring/results_by_source_group.csv
06_truth_scoring/catastrophic_components.csv
06_truth_scoring/clustered_uncertainty.json
08_visual_audit/visual_audit_queue.csv
09_replay/same_run_replay.json
```

If the notebook stops at artefact discovery, share the generated `artifact_discovery.json` and failure JSON. That result will identify exactly which frozen V8 objects are absent or ambiguous.
